In [ ]:
# Phase 5 — Real-Time Report: Tag Watchlist Generator
# Reads ML tables from the lakehouse, identifies the key PI tags
# to watch over the next 24h–14d, and pushes WatchTags() and
# WatchAssets() KQL functions to Eventhouse so the Real-Time
# Dashboard filters to the most important signals.
#
# Schedule: Daily (after Cox + Anomaly notebooks complete)
# Output:   KQL functions in pi-realtime-db Eventhouse

from pyspark.sql import functions as F
from datetime import datetime, timedelta
import requests, json, uuid, re

SCHEMA = "ml"
CT_OFFSET = 5

EH_CLUSTER = "https://trd-8a08ckb2duw406mvvg.z2.kusto.fabric.microsoft.com"
EH_DATABASE = "pi-realtime-db"

run_id = str(uuid.uuid4())
now_utc = datetime.utcnow()
now_ct = now_utc - timedelta(hours=CT_OFFSET)
print(f"Run {run_id[:8]} at {now_ct:%Y-%m-%d %I:%M %p CT}")

In [ ]:
# ── Read latest run from each ML table ──

watchlist = spark.sql(f"""
    SELECT w.* FROM {SCHEMA}.watchlist w
    INNER JOIN (
        SELECT model_name, MAX(model_run_timestamp) AS latest_ts
        FROM {SCHEMA}.watchlist GROUP BY model_name
    ) latest ON w.model_name = latest.model_name
              AND w.model_run_timestamp = latest.latest_ts
""")

pred_long = spark.sql(f"""
    SELECT p.* FROM {SCHEMA}.predictions_longterm p
    INNER JOIN (
        SELECT asset_id, MAX(scoring_timestamp) AS max_ts
        FROM {SCHEMA}.predictions_longterm
        GROUP BY asset_id
    ) latest ON p.asset_id = latest.asset_id
            AND p.scoring_timestamp = latest.max_ts
""")

pred_short = spark.sql(f"""
    SELECT p.* FROM {SCHEMA}.predictions_shortterm p
    INNER JOIN (
        SELECT asset_id, prediction_horizon, label_type, MAX(scoring_timestamp) AS max_ts
        FROM {SCHEMA}.predictions_shortterm
        GROUP BY asset_id, prediction_horizon, label_type
    ) latest ON p.asset_id = latest.asset_id
            AND p.prediction_horizon = latest.prediction_horizon
            AND p.label_type = latest.label_type
            AND p.scoring_timestamp = latest.max_ts
""")

drv_long = spark.sql(f"""
    SELECT * FROM {SCHEMA}.drivers_longterm
    WHERE model_run_timestamp = (
        SELECT MAX(model_run_timestamp) FROM {SCHEMA}.drivers_longterm
    )
""")

drv_short = spark.sql(f"""
    SELECT * FROM {SCHEMA}.drivers_shortterm
    WHERE scored_at = (
        SELECT MAX(scored_at) FROM {SCHEMA}.drivers_shortterm
    )
""")

anom = spark.sql(f"""
     SELECT a.* FROM {SCHEMA}.anomaly_advisories a
     INNER JOIN (
         SELECT asset_id, MAX(scored_at) AS max_ts
         FROM {SCHEMA}.anomaly_advisories
         GROUP BY asset_id
     ) latest ON a.asset_id = latest.asset_id
             AND a.scored_at = latest.max_ts
""")
 
aakr_ep = spark.sql(f"""
     SELECT e.* FROM {SCHEMA}.aakr_episodes e
     INNER JOIN (
         SELECT asset_id, MAX(detected_at) AS max_ts
         FROM {SCHEMA}.aakr_episodes
         GROUP BY asset_id
     ) latest ON e.asset_id = latest.asset_id
             AND e.detected_at = latest.max_ts
""")

print(f"Watchlist:     {watchlist.count()} rows")
print(f"Pred Long:     {pred_long.count()} rows ({pred_long.select('asset_id').distinct().count()} assets)")
print(f"Pred Short:    {pred_short.count()} rows ({pred_short.select('asset_id').distinct().count()} assets)")
print(f"Drivers Long:  {drv_long.count()} rows")
print(f"Drivers Short: {drv_short.count()} rows")

In [ ]:
# ── Normalize tag names and extract signals from all models ──
import pandas as pd

# Load canonical tag descriptions from gold layer
bridge = spark.sql("SELECT Tag, tag_description, eng_units FROM gold.bridge_pi_tag_to_asset").toPandas()
tag_desc_map = dict(zip(bridge['Tag'], bridge['tag_description']))
tag_unit_map = dict(zip(bridge['Tag'], bridge['eng_units']))

def normalize_tag(raw):
    """Convert ML tag names to PiEvents format (RV2:XXX.AG)."""
    tag = re.sub(r'__(avg|delta|std|v|p\d+)\d*h?$', '', str(raw))
    if ':' in tag:
        return tag
    m = re.match(r'^(LG\d)_(.+)_AG$', tag)
    return f"{m.group(1)}:{m.group(2)}.AG" if m else tag

def get_descriptor(tag, fallback=''):
    """Look up tag description from bridge table, fall back to provided."""
    return tag_desc_map.get(tag, fallback) or tag

def get_units(tag, fallback=''):
    """Look up engineering units from bridge table, fall back to provided."""
    return tag_unit_map.get(tag, fallback) or ''

signals = []

# Cox drivers -> hazard ratio signals
for r in drv_long.collect():
    tag = normalize_tag(r['tag_name'])
    signals.append({
        'tag': tag,
        'asset_id': r['asset_id'],
        'descriptor': get_descriptor(tag, r['descriptor']),
        'engineering_units': get_units(tag, r['engineering_units']),
        'source': 'Cox',
        'risk_score': abs(float(r['contribution'] or 0)),
        'current_value': float(r['feature_value'] or 0),
        'hazard_ratio': float(r['hazard_ratio'] or 0),
        'horizon': '14d',
        'action': 'MONITOR' if r['driver_direction'] == 'risk_reducer' else 'INVESTIGATE',
    })

# Short-term drivers -> top SHAP contributors (top 30 per run)
for r in drv_short.filter(F.col('feature_rank') <= 30).collect():
    tag = normalize_tag(r['base_tag'])
    signals.append({
        'tag': tag,
        'asset_id': r['asset_id'],
        'descriptor': get_descriptor(tag),
        'engineering_units': get_units(tag),
        'source': 'GBM',
        'risk_score': abs(float(r['shap_value'] or 0)),
        'current_value': float(r['feature_value'] or 0),
        'hazard_ratio': 0,
        'horizon': r['prediction_horizon'] or '4h',
        'action': 'INVESTIGATE' if r['alert_level'] in ('MEDIUM', 'HIGH', 'CRITICAL') else 'MONITOR',
    })

# Anomaly advisories -> Z-score signals
for r in anom.collect():
    tag = r['Tag']
    signals.append({
        'tag': tag,
        'asset_id': r['asset_id'],
        'descriptor': get_descriptor(tag),
        'engineering_units': get_units(tag),
        'source': 'Anomaly',
        'risk_score': float(r['peak_abs_z'] or 0),
        'current_value': float(r['latest_value'] or 0),
        'hazard_ratio': 0,
        'horizon': '24h',
        'action': 'INVESTIGATE' if r['severity'] in ('HIGH', 'CRITICAL') else 'MONITOR',
    })

# AAKR episodes -> residual signals
for r in aakr_ep.collect():
    tag = r['tag']
    signals.append({
        'tag': tag,
        'asset_id': r['asset_id'],
        'descriptor': get_descriptor(tag),
        'engineering_units': get_units(tag),
        'source': 'AAKR',
        'risk_score': abs(float(r['max_z'] or 0)),
        'current_value': float(r['mean_actual'] or 0),
        'hazard_ratio': 0,
        'horizon': '24h',
        'action': 'INVESTIGATE' if r['severity'] in ('HIGH', 'CRITICAL') else 'MONITOR',
    })

# Watchlist -> already-synthesized cross-model signals
for r in watchlist.collect():
    tag = r['tag_name'] or ''
    if not tag:
        continue
    signals.append({
        'tag': tag,
        'asset_id': r['asset_id'],
        'descriptor': get_descriptor(tag, r['descriptor']),
        'engineering_units': get_units(tag, r['engineering_units']),
        'source': r['model_name'].split('_')[0],
        'risk_score': abs(float(r['risk_contribution'] or 0)),
        'current_value': float(r['current_value'] or 0),
        'hazard_ratio': 0,
        'horizon': f"{r['watch_horizon_days']}d" if r['watch_horizon_days'] else '14d',
        'action': r['recommended_action'] or 'MONITOR',
    })

df_signals = pd.DataFrame(signals)
df_signals = df_signals[df_signals['tag'].str.contains(':', na=False)]

# Verify descriptor coverage
has_desc = (df_signals['descriptor'] != df_signals['tag']).sum()
print(f"Total raw signals: {len(df_signals)} across {df_signals['tag'].nunique()} unique tags")
print(f"Descriptor coverage: {has_desc}/{len(df_signals)} ({has_desc/len(df_signals)*100:.0f}%)")

In [ ]:
# ── Score composite watch priority per tag ──

tag_agg = df_signals.groupby(['tag', 'asset_id']).agg(
    model_count=('source', 'nunique'),
    sources=('source', lambda x: ','.join(sorted(set(x)))),
    max_risk=('risk_score', 'max'),
    best_action=('action', lambda x: 'INVESTIGATE' if 'INVESTIGATE' in x.values else 'MONITOR'),
    shortest_horizon=('horizon', 'first'),
    current_value=('current_value', 'last'),
    hazard_ratio=('hazard_ratio', 'max'),
).reset_index()

# Fill in descriptors and units from any source that had them
desc_map = df_signals[df_signals['descriptor'] != ''].drop_duplicates('tag').set_index('tag')['descriptor'].to_dict()
unit_map = df_signals[df_signals['engineering_units'] != ''].drop_duplicates('tag').set_index('tag')['engineering_units'].to_dict()
tag_agg['descriptor'] = tag_agg['tag'].map(desc_map).fillna('')
tag_agg['engineering_units'] = tag_agg['tag'].map(unit_map).fillna('')

def priority(row):
    if row['model_count'] >= 3 or (row['max_risk'] > 50 and row['model_count'] >= 2):
        return 'CRITICAL'
    if row['model_count'] >= 2 or row['max_risk'] > 10:
        return 'HIGH'
    if row['max_risk'] > 3 or row['best_action'] == 'INVESTIGATE':
        return 'MEDIUM'
    return 'LOW'

tag_agg['watch_priority'] = tag_agg.apply(priority, axis=1)

priority_order = {'CRITICAL': 0, 'HIGH': 1, 'MEDIUM': 2, 'LOW': 3}
tag_agg['_sort'] = tag_agg['watch_priority'].map(priority_order)
tag_agg = tag_agg.sort_values(['_sort', 'max_risk'], ascending=[True, False]).drop(columns='_sort')

print(f"Watch priority breakdown:")
print(tag_agg['watch_priority'].value_counts().to_string())
print(f"\nTop 15 tags to watch:")
for _, r in tag_agg.head(15).iterrows():
    print(f"  {r['watch_priority']:8s} | {r['tag']:30s} | {r['sources']:20s} | risk={r['max_risk']:.2f}")

In [ ]:
# ── Build asset-level summary ──

assets = []
for r in pred_long.collect():
    st = pred_short.filter(
        (F.col('asset_id') == r['asset_id']) &
        (F.col('prediction_horizon') == '4h') &
        (F.col('label_type') == 'stop')
    ).first()

    assets.append({
        'asset_id': r['asset_id'],
        'risk_level': r['risk_level'],
        'cox_survival_7d': round(float(r['survival_probability_7d'] or 0), 3),
        'cox_survival_14d': round(float(r['survival_probability_14d'] or 0), 3),
        'short_term_stop_4h': round(float(st['stop_probability']) if st else 0, 3),
        'short_term_alert': st['alert_level'] if st else 'UNKNOWN',
    })

df_assets = pd.DataFrame(assets).drop_duplicates('asset_id')
print("Asset summary:")
for _, a in df_assets.iterrows():
    print(f"  {a['asset_id']:35s} | {a['risk_level']:8s} | "
          f"7d={a['cox_survival_7d']:.1%} | 14d={a['cox_survival_14d']:.1%} | "
          f"4h stop={a['short_term_stop_4h']:.1%}")

In [ ]:
# ── Push WatchTags() and WatchAssets() functions to Eventhouse ──

token = notebookutils.credentials.getToken(EH_CLUSTER)
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
mgmt_url = f"{EH_CLUSTER}/v1/rest/mgmt"

def kql_escape(s):
    return str(s).replace('\\', '\\\\').replace("'", "\\'").replace('"', '\\"')

# ── WatchTags() ──
rows = []
for _, r in tag_agg.iterrows():
    rows.append(
        f'    "{kql_escape(r["tag"])}", "{kql_escape(r["asset_id"])}", '
        f'"{kql_escape(r["descriptor"])}", "{kql_escape(r["engineering_units"])}", '
        f'"{r["watch_priority"]}", "{r["shortest_horizon"]}", '
        f'"{kql_escape(r["sources"])}", real({r["max_risk"]:.4f}), '
        f'real({r["current_value"]:.4f}), real({r["hazard_ratio"]:.4f}), '
        f'int({r["model_count"]}), "{r["best_action"]}"'
    )
rows_kql = ",\n".join(rows)

watch_fn = (
    f'.create-or-alter function with '
    f'(docstring = "ML-driven tag watchlist — updated {now_ct:%Y-%m-%d %H:%M CT}", folder = "ml") '
    f'WatchTags() {{\n'
    f'    datatable(tag:string, asset_id:string, descriptor:string, engineering_units:string,\n'
    f'              watch_priority:string, watch_horizon:string, signal_sources:string,\n'
    f'              risk_score:real, current_value:real, hazard_ratio:real,\n'
    f'              model_agreement_count:int, recommended_action:string)\n'
    f'    [\n{rows_kql}\n    ]\n}}'
)

# ── WatchAssets() ──
asset_rows = []
for _, a in df_assets.iterrows():
    asset_rows.append(
        f'    "{kql_escape(a["asset_id"])}", "{a["risk_level"]}", '
        f'real({a["cox_survival_7d"]}), real({a["cox_survival_14d"]}), '
        f'real({a["short_term_stop_4h"]}), "{a["short_term_alert"]}", '
        f'datetime({now_ct:%Y-%m-%dT%H:%M:%SZ})'
    )
asset_rows_kql = ",\n".join(asset_rows)

asset_fn = (
    f'.create-or-alter function with '
    f'(docstring = "Asset-level risk summary — updated {now_ct:%Y-%m-%d %H:%M CT}", folder = "ml") '
    f'WatchAssets() {{\n'
    f'    datatable(asset_id:string, risk_level:string, cox_survival_7d:real, cox_survival_14d:real,\n'
    f'              short_term_stop_prob_4h:real, short_term_alert:string, scored_at:datetime)\n'
    f'    [\n{asset_rows_kql}\n    ]\n}}'
)

# Execute both management commands
for name, cmd in [("WatchTags", watch_fn), ("WatchAssets", asset_fn)]:
    body = {"csl": cmd, "db": EH_DATABASE}
    resp = requests.post(mgmt_url, headers=headers, json=body)
    if resp.status_code == 200:
        print(f"  WatchTags:   {tag_agg['watch_priority'].value_counts().to_dict()}")
        print(f"  WatchAssets: {len(df_assets)} assets")
    else:
        print(f"  {name}() FAILED: {resp.status_code} -- {resp.text[:300]}")

In [ ]:
# ── Summary ──
print(f"\n{'='*60}")
print(f"Phase5-RealTime-Report complete - run {run_id[:8]}")
print(f"{'='*60}")
print(f"Tags:   {len(tag_agg)} ({tag_agg['watch_priority'].value_counts().to_dict()})")
print(f"Assets: {len(df_assets)}")
print(f"\nDashboard tiles can now use WatchTags() and WatchAssets().")
print(f"\nExample KQL — critical/high tags with descriptions:")
print(f"""
PiEvents
| where Ts > ago(24h)
| where Tag in (WatchTags() | where watch_priority in ("CRITICAL","HIGH") | project tag)
| where not(Questionable)
| lookup (WatchTags() | project tag, descriptor) on $left.Tag == $right.tag
| summarize avg(toreal(Value)) by bin(Ts, 5m), descriptor
| render timechart
""")
print(f"Example KQL — current values with descriptions:")
print(f"""
let wt = WatchTags();
let current = PiEvents
| where Ts > ago(1h) and not(Questionable)
| where Tag in (wt | project tag)
| summarize CurrentValue=avg(toreal(Value)), LastReading=max(Ts) by Tag;
current
| join kind=inner wt on $left.Tag == $right.tag
| project Sensor=descriptor, CurrentValue=round(CurrentValue,2),
          Priority=watch_priority, Sources=signal_sources,
          Units=engineering_units, Action=recommended_action
| order by Priority asc
""")